# Phase 2C — Stratified split and training-only preprocessing

Scenario: subscription prediction **before campaign initiation**. This notebook prepares unfitted preprocessing factories and reproducible partitions, then performs technical tests only on a 400-record development subset. **No models, tuning, resampling or predictive evaluation.** The full development set is never preprocessed; the held-out features are never transformed or profiled for design choices. Required class counts are reported solely to verify stratification.

Reviewed README, project DOCX, Phase 2A decision documents, Phase 2B notebook/summary and cleaned CSV. Phase 2B's as-of concerns remain; the user now explicitly approves retaining these features as **Provisional**. Preprocessing isolation does not establish feature-timing or client-disjoint leakage safety.

## Colab instructions

Upload `03_train_test_preprocessing.ipynb`, `preprocessing.py`, `02_cleaned_dataset.csv` and original `dataset.csv` together. The raw file is read only to audit Phase 2B provenance; cleaning is not rerun or exported. Optionally upload `tests/test_preprocessing.py` to run the standalone synthetic unit tests. Set `PROJECT_DIR` to a mounted Drive project directory if desired. Standard Colab provides NumPy, pandas and scikit-learn; this code requires **scikit-learn >=1.2** (`sparse_output` support). Outputs are split-index JSON and a Markdown summary only, never scaled/encoded CSVs or fitted production artifacts. Download them through Colab's Files panel if needed. Rerunning regenerates the same split positions; input SHA-256 prevents applying those positions to a different CSV.


In [1]:
from pathlib import Path
import sys, json, hashlib, warnings
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from IPython.display import display
PROJECT_DIR = Path.cwd()  # or Path('/content/drive/MyDrive/bank-marketing-ml')
required = ['preprocessing.py', '02_cleaned_dataset.csv', 'dataset.csv']
if any(not (PROJECT_DIR/n).is_file() for n in required):
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Place all required files in PROJECT_DIR') from exc
    files.upload()
if any(not (PROJECT_DIR/n).is_file() for n in required):
    raise FileNotFoundError('Missing required files; see Colab instructions')
sys.path.insert(0, str(PROJECT_DIR))
from preprocessing import (CATEGORICAL, NUMERICAL, RECENCY, FEATURES, EXCLUDED,
                           ConditionalRecencyScaler, make_preprocessor, make_cv)
clean_path = PROJECT_DIR/'02_cleaned_dataset.csv'
raw_path = PROJECT_DIR/'dataset.csv'
input_hashes = {p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in [raw_path,clean_path]}
clean = pd.read_csv(clean_path, float_precision='round_trip')
raw = pd.read_csv(raw_path, float_precision='round_trip')
clean_snapshot = clean.copy(deep=True)
raw_snapshot = raw.copy(deep=True)
print('scikit-learn:', sklearn.__version__)
print('Input hashes:', input_hashes)


scikit-learn: 1.8.0
Input hashes: {'dataset.csv': '81fd5eb4b659dddbc704edd90376c32714c3f6a43b887cb686762903bc793f0d', '02_cleaned_dataset.csv': 'c1a19ca755279a5f22ba22e06b8d5ca22797d9b6351c69fe4cfafd2b47041d49'}


## 1. Validate the existing cleaned artifact, without re-cleaning

All original columns define duplicate identity; the approved mask identifies the 12 rows removed in Phase 2B solely for comparison. The existing cleaned file must equal the approved projection of retained original rows, in order. No additional rows, categories or features are removed. A discrepancy raises an error before any split/fitting. Validate full-input schema/domain/provenance as required, then freeze preprocessing design; do not use this check to optimize against held-out distributions.


In [2]:
assert clean.shape == (41176,16)
assert clean.columns.tolist() == FEATURES + ['y']
assert not set(EXCLUDED).intersection(clean.columns)
assert not clean.isna().any().any()
assert clean.y.isin([0,1]).all()
assert clean.recency_available.isin([0,1]).all()
assert np.isfinite(clean[NUMERICAL+RECENCY].to_numpy(dtype=float)).all()
assert clean.recency_days.ge(0).all() and clean.recency_days.mod(1).eq(0).all()
assert clean.loc[clean.recency_available.eq(0),'recency_days'].eq(0).all()
assert raw.shape == (41188,20)
keep = ~raw.duplicated(keep='first')  # provenance check only, never overwrite either CSV
assert (~keep).sum() == 12 and keep.sum() == len(clean)
retained = raw.loc[keep].reset_index(drop=True)
for c in [c for c in clean if c not in ['default','housing','loan']+RECENCY]:
    pd.testing.assert_series_equal(clean[c], retained[c], check_exact=True)
for c in ['default','housing','loan']:
    expected = retained[c].map({-1:'unknown',0:'no',1:'yes'})
    pd.testing.assert_series_equal(clean[c], expected, check_exact=True)
np.testing.assert_array_equal(clean.recency_available, retained.pdays.ne(999).astype(int))
np.testing.assert_array_equal(clean.recency_days, np.where(retained.pdays.eq(999),0,retained.pdays))
unknown_counts = {c:int(clean[c].eq('unknown').sum()) for c in CATEGORICAL}
assert unknown_counts == dict(job=330,marital=80,education=1730,default=8596,housing=990,loan=990)
assert ((clean.previous>0)&(clean.recency_available==0)).sum() == 4110
print('Cleaned artifact and approved duplicate removal verified. No changes made.')
print('Explicit unknown counts:', unknown_counts)


Cleaned artifact and approved duplicate removal verified. No changes made.
Explicit unknown counts: {'job': 330, 'marital': 80, 'education': 1730, 'default': 8596, 'housing': 990, 'loan': 990}


## 2. Reproducible stratified 80/20 partition

Indices are **zero-based positional rows of the cleaned CSV**, not raw-file line numbers or customer IDs. The saved JSON records their ordered arrays, source hash, row count and splitting parameters. Reload by checking the hash first and selecting with `.iloc`. The held-out subset remains reserved: its class counts below are only a requested stratification check, not a modeling result. No preprocessing is fitted on development or held-out in this section.


In [3]:
positions = np.arange(len(clean), dtype=int)
y = clean['y']
dev_positions, test_positions = train_test_split(positions, test_size=0.2, random_state=42, stratify=y)
repeat_dev, repeat_test = train_test_split(positions, test_size=0.2, random_state=42, stratify=y)
np.testing.assert_array_equal(dev_positions, repeat_dev)
np.testing.assert_array_equal(test_positions, repeat_test)
assert len(dev_positions)==32940 and len(test_positions)==8236
assert not set(dev_positions).intersection(test_positions)
np.testing.assert_array_equal(np.sort(np.r_[dev_positions,test_positions]),positions)
X_dev = clean.iloc[dev_positions][FEATURES].copy()
y_dev = y.iloc[dev_positions].copy()
# Intentionally do not create/preprocess X_test or inspect held-out feature distributions.
split_rows=[]
for name,idx in [('development',dev_positions),('held_out',test_positions)]:
    counts=y.iloc[idx].value_counts().reindex([0,1],fill_value=0)
    split_rows.append({'partition':name,'n':len(idx),'no':int(counts[0]),'yes':int(counts[1]),
                       'no_percent':float(counts[0]/len(idx)*100), 'yes_percent':float(counts[1]/len(idx)*100)})
split_table=pd.DataFrame(split_rows).set_index('partition')
display(split_table)
manifest={'input_file':clean_path.name,'input_sha256':input_hashes[clean_path.name],
          'index_type':'zero-based cleaned CSV row positions; ordered arrays',
          'row_count':len(clean),'test_size':0.2,'random_state':42,'stratify':'y',
          'sklearn_version':sklearn.__version__,
          'development_indices':dev_positions.tolist(),'held_out_indices':test_positions.tolist()}
manifest_path=PROJECT_DIR/'03_split_indices.json'
manifest_path.write_text(json.dumps(manifest, separators=(',',':'))+'\n')
restored=json.loads(manifest_path.read_text())
assert restored['input_sha256']==hashlib.sha256(clean_path.read_bytes()).hexdigest()
np.testing.assert_array_equal(restored['development_indices'],dev_positions)
np.testing.assert_array_equal(restored['held_out_indices'],test_positions)
print('Reproducibility, coverage, disjointness and index persistence passed.')


Reproducibility, coverage, disjointness and index persistence passed.


,n,no,yes,no_percent,yes_percent
partition,,,,,
development,32940,29229,3711,88.734062,11.265938
held_out,8236,7308,928,88.732394,11.267606


## 3. Unfitted model-specific factories and five-fold configuration

`make_preprocessor` returns an unfitted sklearn Pipeline containing a ColumnTransformer. Six categorical columns use full OneHotEncoder (`drop=None`, `handle_unknown='ignore'`); category vocabularies are learned only from the fit input. A documented `unknown` seen during fit gets its own dummy. If any category, even `unknown`, is absent from that training fold, it is unseen and maps to an all-zero block at transform time. No categories are injected based on test data. This preserves source unknowns without confusing them with mode imputation. Unseen-category rates should be monitored later. No grouping, reference-level drop, label/ordinal/target encoding is applied.

LR: StandardScaler for seven quantitative columns excluding recency_days; conditional scaler for the two recency inputs. Trees/RF: identical categorical handling; quantitative values and both recency fields pass through unchanged. Dense outputs are reasonable for these low-cardinality predictors; no predictive estimator is present.

Order is categorical blocks → seven ordinary numerical columns → recency_days and recency_available for every configuration. Names come from `get_feature_names_out`. For the same fitted instance ordering is fixed; vocabularies/width may differ across independently fitted CV folds when rare categories are absent. That is expected and valid: each future estimator must stay coupled to its own fold's preprocessor.

**Future use:** build a fresh complete estimator Pipeline with `make_preprocessor(...)` as its preprocessing step, then pass the *untransformed development data* and `make_cv()` to the model-development procedure. Never feed the whole-development transformed matrix into CV. This notebook does not run that future procedure.


In [4]:
configurations = {name:make_preprocessor(name) for name in ['logistic_regression','decision_tree','random_forest']}
cv=make_cv()
assert cv.n_splits==5 and cv.shuffle and cv.random_state==42
for p in configurations.values():
    assert not hasattr(p.named_steps['preprocessor'],'transformers_')
    clone(p)
print('Three fresh unfitted factories; StratifiedKFold(n_splits=5, shuffle=True, random_state=42).')


Three fresh unfitted factories; StratifiedKFold(n_splits=5, shuffle=True, random_state=42).


## 4. Conditional recency scaling mathematics and edge cases

For a training subset, let $A=\{i:a_i=1\}$ where $a_i$ is recency_available and $d_i$ recency_days. Estimate $\mu_A=|A|^{-1}\sum_{i\in A}d_i$ and $\sigma_A=\sqrt{|A|^{-1}\sum_{i\in A}(d_i-\mu_A)^2}$ (population standard deviation, ddof=0). Output $z_i=(d_i-\mu_A)/s_A$ if available, otherwise $z_i=0$, and output $a_i$ unchanged. Here $s_A=\sigma_A$ when positive, else 1.

Placeholders contribute to neither mean nor variance; ordinary StandardScaler across recency_days would let the overwhelmingly unavailable values distort both. With no available training samples, use mean=0 and scale=1 and emit a warning: no recency distribution was learned. With one available sample or zero variance, use observed mean and unit scale, warning as well. These deterministic safeguards prevent division by zero; they do not imply adequate evidence. Validation/test values always use the training-fitted parameters, even if far outside the training range. Negative/nonfinite recency, invalid indicators, nonzero unavailable placeholders and wrong column order raise errors, not corrections.

A transformed actual-zero value can coincide numerically with an unavailable placeholder in a degenerate training set; the separate availability bit still distinguishes the cases. Synthetic assertions below cover genuine zero days, placeholder exclusion, no-refit behavior, cloneability, invalid input and every fallback. They are technical examples, not invented customer records added to the dataset.


In [5]:
synthetic = pd.DataFrame([[0,0]]*100 + [[0,1],[2,1],[4,1]],columns=RECENCY)
s=ConditionalRecencyScaler().fit(synthetic)
assert s.n_available_==3 and s.mean_==2
np.testing.assert_allclose(s.scale_,np.std([0,2,4],ddof=0))
np.testing.assert_allclose(s.var_,np.var([0,2,4],ddof=0))
state=(s.mean_,s.var_,s.scale_,s.n_available_)
probe=pd.DataFrame([[0,0],[0,1],[1000,1]],columns=RECENCY)
t=s.transform(probe)
np.testing.assert_array_equal(t[0],[0,0])
assert t[1,1]==1 and t[1,0]!=t[0,0]
assert (s.mean_,s.var_,s.scale_,s.n_available_)==state
for rows,n,mean in [([[0,0],[0,0]],0,0),([[0,0],[4,1]],1,4),([[3,1],[3,1]],2,3),([[0,1],[0,1]],2,0)]:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        edge=ConditionalRecencyScaler().fit(pd.DataFrame(rows,columns=RECENCY))
    assert caught and edge.n_available_==n and edge.mean_==mean and edge.scale_==1
    out=edge.transform(probe)
    assert np.isfinite(out).all() and out[0,1]==0 and out[1,1]==1
for invalid in [[[1,0]],[[-1,1]],[[0,2]],[[np.nan,1]],[[np.inf,1]]]:
    try:ConditionalRecencyScaler().fit(pd.DataFrame(invalid,columns=RECENCY))
    except ValueError:pass
    else:raise AssertionError('Invalid input accepted')
print('Explicit conditional-scaler assertions passed; parameters ignore placeholders and validation.')


Explicit conditional-scaler assertions passed; parameters ignore placeholders and validation.


## 5. Limited development-only technical tests

Select 400 development records reproducibly, then split those into 300 fit / 100 validation records. This subset is for software checks only; it is not a new test set or a model-selection experiment. Fit clones, never the factory configurations. The held-out feature rows are not accessed. No threshold/choice is optimized from validation outcomes; labels only stratify subset allocation and fold membership.

Check category vocabularies against fit rows, StandardScaler means against fit rows, conditional statistics against available fit recencies, transformed feature names/order/finite values, and unchanged parameters after validation. Inject a synthetic unseen string and explicit unknowns into a *copy* of a development validation row to test encoder behavior. No source/development observations are permanently changed.


In [6]:
subset_positions,_=train_test_split(dev_positions,train_size=400,random_state=42,stratify=y.iloc[dev_positions])
fit_positions,val_positions=train_test_split(subset_positions,test_size=100,random_state=42,stratify=y.iloc[subset_positions])
assert set(subset_positions).issubset(set(dev_positions)) and not set(subset_positions).intersection(test_positions)
fit_X=clean.iloc[fit_positions][FEATURES].copy()
val_X=clean.iloc[val_positions][FEATURES].copy()
assert fit_X.recency_available.eq(1).sum()>1

def signature(p):
    c=p.named_steps['preprocessor']
    result={'categories':[v.tolist() for v in c.named_transformers_['categorical'].categories_]}
    if hasattr(c.named_transformers_['numerical'], 'mean_'):
        num=c.named_transformers_['numerical']
        result['numerical']=[num.mean_.tolist(),num.var_.tolist(),np.asarray(num.n_samples_seen_).tolist()]
    rec=c.named_transformers_['recency']
    if isinstance(rec,ConditionalRecencyScaler):
        result['recency']=[rec.mean_,rec.var_,rec.scale_,rec.n_available_]
    return result

technical_rows=[]
for name,factory in configurations.items():
    fitted=clone(factory).fit(fit_X)
    before=signature(fitted)
    train_out=fitted.transform(fit_X)
    val_out=fitted.transform(val_X)
    names=fitted.get_feature_names_out()
    assert train_out.shape[1]==val_out.shape[1]==len(names) and len(set(names))==len(names)
    assert np.isfinite(train_out).all() and np.isfinite(val_out).all()
    ct=fitted.named_steps['preprocessor']; enc=ct.named_transformers_['categorical']
    assert enc.drop is None and enc.handle_unknown=='ignore'
    for c,levels in zip(CATEGORICAL,enc.categories_):
        np.testing.assert_array_equal(levels,np.sort(fit_X[c].unique()))
    # Each documented unknown observed in training gets a distinct one-hot column.
    unknown_probe=val_X.iloc[:1].copy()
    unknown_probe[CATEGORICAL]='unknown'
    categorical_out=enc.transform(unknown_probe[CATEGORICAL])
    start=0
    for levels in enc.categories_:
        block=categorical_out[0,start:start+len(levels)]
        if 'unknown' in levels:
            assert block.sum()==1 and block[list(levels).index('unknown')]==1
        else:assert block.sum()==0
        start+=len(levels)
    unseen=val_X.iloc[:1].copy(); unseen['job']='__synthetic_unseen_job__'
    unseen_out=fitted.transform(unseen)
    assert np.isfinite(unseen_out).all() and unseen_out.shape[1]==len(names)
    assert unseen_out[0,:len(enc.categories_[0])].sum()==0
    if name=='logistic_regression':
        np.testing.assert_allclose(ct.named_transformers_['numerical'].mean_,fit_X[NUMERICAL].mean())
        rec=ct.named_transformers_['recency']
        observed=fit_X.loc[fit_X.recency_available.eq(1),'recency_days'].to_numpy()
        assert rec.n_available_==len(observed)
        np.testing.assert_allclose([rec.mean_,rec.var_],[observed.mean(),observed.var(ddof=0)])
        np.testing.assert_array_equal(val_out[:,-1],val_X.recency_available)
        assert (val_out[val_X.recency_available.eq(0),-2]==0).all()
    else:
        np.testing.assert_array_equal(val_out[:,-9:],val_X[NUMERICAL+RECENCY].to_numpy())
    assert signature(fitted)==before, 'Validation changed preprocessing parameters'
    technical_rows.append({'configuration':name,'fit_rows':len(fit_X),'validation_rows':len(val_X),'output_features':len(names),'status':'PASS'})
    print(name, 'feature names:',names.tolist())
display(pd.DataFrame(technical_rows))


logistic_regression feature names: ['categorical__job_admin.', 'categorical__job_blue-collar', 'categorical__job_entrepreneur', 'categorical__job_housemaid', 'categorical__job_management', 'categorical__job_retired', 'categorical__job_self-employed', 'categorical__job_services', 'categorical__job_student', 'categorical__job_technician', 'categorical__job_unemployed', 'categorical__marital_divorced', 'categorical__marital_married', 'categorical__marital_single', 'categorical__marital_unknown', 'categorical__education_basic.4y', 'categorical__education_basic.6y', 'categorical__education_basic.9y', 'categorical__education_high.school', 'categorical__education_professional.course', 'categorical__education_university.degree', 'categorical__education_unknown', 'categorical__default_no', 'categorical__default_unknown', 'categorical__housing_no', 'categorical__housing_unknown', 'categorical__housing_yes', 'categorical__loan_no', 'categorical__loan_unknown', 'categorical__loan_yes', 'numerical_

,configuration,fit_rows,validation_rows,output_features,status
0,logistic_regression,300,100,39,PASS
1,decision_tree,300,100,39,PASS
2,random_forest,300,100,39,PASS


## 6. Five-fold compatibility on the same small development subset

Only software compatibility is exercised here: cloned preprocessing components are fitted on each fold's 320 training rows and transform its 80 validation rows. This is **not** cross-validated predictive evaluation. Every fitted vocabulary/scaling statistic is checked against that fold's training values and frozen through transform. No complete-development preprocessing, predictive estimator or GridSearchCV is run. Future model training must use new unfitted factories rather than these disposable technical instances.


In [7]:
subset_X=clean.iloc[subset_positions][FEATURES]
subset_y=y.iloc[subset_positions]
fold_rows=[]
for fold,(train_local,val_local) in enumerate(cv.split(subset_X,subset_y),start=1):
    assert not set(train_local).intersection(val_local)
    fold_X=subset_X.iloc[train_local]; fold_val=subset_X.iloc[val_local]
    for name,factory in configurations.items():
        p=clone(factory).fit(fold_X)
        before=signature(p)
        out=p.transform(fold_val)
        assert np.isfinite(out).all() and out.shape[1]==len(p.get_feature_names_out())
        ct=p.named_steps['preprocessor']
        for c,levels in zip(CATEGORICAL,ct.named_transformers_['categorical'].categories_):
            np.testing.assert_array_equal(levels,np.sort(fold_X[c].unique()))
        if name=='logistic_regression':
            np.testing.assert_allclose(ct.named_transformers_['numerical'].mean_,fold_X[NUMERICAL].mean())
            actual=fold_X.loc[fold_X.recency_available.eq(1),'recency_days']
            assert ct.named_transformers_['recency'].n_available_==len(actual)
            np.testing.assert_allclose(ct.named_transformers_['recency'].mean_,actual.mean())
            np.testing.assert_allclose(ct.named_transformers_['recency'].var_,actual.var(ddof=0))
        assert signature(p)==before
    fold_rows.append({'fold':fold,'training_rows':len(train_local),'validation_rows':len(val_local),'status':'PASS'})
display(pd.DataFrame(fold_rows))
for factory in configurations.values():
    assert not hasattr(factory.named_steps['preprocessor'],'transformers_')
print('All factories remain unfitted. No fitted artifact is exported or reused.')


All factories remain unfitted. No fitted artifact is exported or reused.


,fold,training_rows,validation_rows,status
0,1,320,80,PASS
1,2,320,80,PASS
2,3,320,80,PASS
3,4,320,80,PASS
4,5,320,80,PASS


## 7. Feature availability and unresolved decisions

All 15 predictors remain, as approved. None has timestamped evidence establishing campaign-start availability, so **all are Provisional for timing**, even if their measurement meaning is documented. No claim of complete leakage freedom is made.


In [8]:
availability_rows=[]
for feature in FEATURES:
    if feature in ['previous','recency_available','recency_days']:
        assumption='Historical campaigns only, known at campaign start'
        risk='Current/future-history updates; recency may be calculated at later contact date'
        evidence='Timestamped historical snapshot, campaign-start date, pdays reference date and history scope'
    elif feature in ['emp.var.rate','cons.price.idx','cons.conf.idx','euribor3m','nr.employed']:
        assumption='Published economic value available by campaign start'
        risk='Later call-period value or revised publication; shared temporal context'
        evidence='Campaign-start timestamp, publication/revision vintage and as-of join or approved lag'
    else:
        assumption='Customer-record value recorded before campaign start'
        risk='Retrospective demographic/financial update after campaign or subscription'
        evidence='Customer attribute/status effective date and campaign-start snapshot'
    availability_rows.append({'feature':feature,'assumed_availability':assumption,'potential_leakage_risk':risk,
                              'required_confirmation':evidence,'status':'Provisional'})
availability=pd.DataFrame(availability_rows)
display(availability)


,feature,assumed_availability,potential_leakage_risk,required_confirmation,status
0,age,Customer-record value recorded before campaign...,Retrospective demographic/financial update aft...,Customer attribute/status effective date and c...,Provisional
1,job,Customer-record value recorded before campaign...,Retrospective demographic/financial update aft...,Customer attribute/status effective date and c...,Provisional
2,marital,Customer-record value recorded before campaign...,Retrospective demographic/financial update aft...,Customer attribute/status effective date and c...,Provisional
3,education,Customer-record value recorded before campaign...,Retrospective demographic/financial update aft...,Customer attribute/status effective date and c...,Provisional
4,default,Customer-record value recorded before campaign...,Retrospective demographic/financial update aft...,Customer attribute/status effective date and c...,Provisional
5,housing,Customer-record value recorded before campaign...,Retrospective demographic/financial update aft...,Customer attribute/status effective date and c...,Provisional
6,loan,Customer-record value recorded before campaign...,Retrospective demographic/financial update aft...,Customer attribute/status effective date and c...,Provisional
7,previous,"Historical campaigns only, known at campaign s...",Current/future-history updates; recency may be...,"Timestamped historical snapshot, campaign-star...",Provisional
8,emp.var.rate,Published economic value available by campaign...,Later call-period value or revised publication...,"Campaign-start timestamp, publication/revision...",Provisional
9,cons.price.idx,Published economic value available by campaign...,Later call-period value or revised publication...,"Campaign-start timestamp, publication/revision...",Provisional


Discussion requiring team confirmation:

1. Are provisional features available before campaign initiation? Retention is approved for this technical phase, not proof of timing safety. Economic publication vintages and recency reference dates are especially important; label outcome window also needs verification.
2. Should rare categories be grouped? Keep full one-hot as approved. Some folds may omit rare levels; consider grouping/reference-level conventions only after explicit approval. Full one-hot coefficients are relative to a coding/regularization convention; predictive performance is not assessed here.
3. Should highly correlated economic indicators remain together? Retained per approval; Phase 2A correlations suggest redundancy, not that a feature is useless. Later development-only comparisons and regularization may be appropriate; no feature selection occurs here.
4. Without customer identifiers and historical timestamps, random stratification cannot certify client-disjoint or future-period generalization. Repeated predictor profiles are not reliable IDs; exact duplicate removal does not remove all repeated-client risk. Obtain identifiers/dates for a future approved group/time validation design; do not substitute row order for chronology.
5. The dataset has previously been viewed during EDA; report the future held-out set as unused for fitting/tuning, rather than claiming its observations were never viewed. No held-out feature distributions or performance were used here.

The supplied pasted request ended during section 9, item 4. Any missing requirements remain unverified; the files here implement only the complete instructions received.


In [9]:
for path in [raw_path,clean_path]:
    assert hashlib.sha256(path.read_bytes()).hexdigest()==input_hashes[path.name]
pd.testing.assert_frame_equal(clean,clean_snapshot,check_exact=True)
pd.testing.assert_frame_equal(raw,raw_snapshot,check_exact=True)
print('Both datasets unchanged; no encoded/scaled CSVs or predictive models created.')


Both datasets unchanged; no encoded/scaled CSVs or predictive models created.
